In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/NoonGil'
os.makedirs(DRIVE_BASE, exist_ok=True)
print("드라이브 마운트 완료")
print(f"저장 경로: {DRIVE_BASE}")

Mounted at /content/drive
드라이브 마운트 완료
저장 경로: /content/drive/MyDrive/NoonGil


In [2]:
!pip install ultralytics -q

import ultralytics
ultralytics.checks()
print("YOLOv8 설치 완료")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.1/112.6 GB disk)
YOLOv8 설치 완료


In [3]:
!pip install roboflow -q
from roboflow import Roboflow
import shutil, os

API_KEY = "vgvnQD3ktPGMXOXsiCG7"  # 본인 API key 입력
rf = Roboflow(api_key=API_KEY)
ws = "s-workspace-6sd3n"

# NoonGil v4
rf.workspace(ws).project("noongil-barrierfree-ai").version(4).download(
    "yolov8", location="/content/datasets/noongil")

# ramp v1
rf.workspace(ws).project("ramp-udoeh-gl36m").version(1).download(
    "yolov8", location="/content/datasets/ramp")

# pothole v3
rf.workspace(ws).project("pothole-vhmow-jp2rw").version(3).download(
    "yolov8", location="/content/datasets/pothole")

print("전체 다운로드 완료")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 83.2 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/datasets/noongil in yolov8:: 100%|██████████| 1789/1789 [00:00<00:00, 3513.14it/s]

loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /content/datasets/ramp in yolov8:: 100%|██████████| 229/229 [00:00<00:00, 2289.03it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/datasets/pothole in yolov8:: 100%|██████████| 829/829 [00:00<00:00, 6895.28it/s]

전체 다운로드 완료


In [4]:
from pathlib import Path

PUBLIC_CLASS_MAP = {
    'ramp': {
        '0': '9',   # ramp(0)   → ramp(9)
        '1': '10',  # stairs(1) → step(10)
    },
    'pothole': {'0': '8'},  # pothole(0) → pavement_damage(8)
}

def remap_label(src_label_path, dst_label_path, class_map):
    with open(src_label_path, 'r') as f:
        lines = f.readlines()
    remapped = []
    for line in lines:
        parts = line.strip().split()
        if not parts:
            continue
        old_id = parts[0]
        new_id = class_map.get(old_id)
        if new_id is None:
            continue
        remapped.append(f"{new_id} {' '.join(parts[1:])}\n")
    if remapped:
        with open(dst_label_path, 'w') as f:
            f.writelines(remapped)

def copy_split(src_base, dst_base, split, class_map=None, prefix=''):
    src_img = Path(src_base) / split / 'images'
    src_lbl = Path(src_base) / split / 'labels'
    dst_img = Path(dst_base) / split / 'images'
    dst_lbl = Path(dst_base) / split / 'labels'

    dst_img.mkdir(parents=True, exist_ok=True)
    dst_lbl.mkdir(parents=True, exist_ok=True)

    if not src_img.exists():
        print(f"  [{split}] 폴더 없음, 스킵")
        return 0

    images = list(src_img.glob('*.jpg')) + \
             list(src_img.glob('*.png')) + \
             list(src_img.glob('*.jpeg'))
    count = 0
    for img_path in images:
        stem = img_path.stem
        new_stem = f"{prefix}_{stem}"
        shutil.copy(img_path, dst_img / f"{new_stem}{img_path.suffix}")
        lbl_path = src_lbl / f"{stem}.txt"
        dst_lbl_path = dst_lbl / f"{new_stem}.txt"
        if lbl_path.exists():
            if class_map:
                remap_label(lbl_path, dst_lbl_path, class_map)
            else:
                shutil.copy(lbl_path, dst_lbl_path)
        count += 1
    print(f"  [{split}] {count}장 완료")
    return count

print("함수 정의 완료")

함수 정의 완료


In [5]:
MERGED = '/content/datasets/merged'

if os.path.exists(MERGED):
    shutil.rmtree(MERGED)
    print("기존 merged 폴더 삭제 완료")

print("=== NoonGil ===")
for split in ['train', 'valid', 'test']:
    copy_split('/content/datasets/noongil', MERGED, split,
               class_map=None, prefix='noongil')

print("\n=== ramp (ramp→9, stairs→10) ===")
for split in ['train', 'valid', 'test']:
    copy_split('/content/datasets/ramp', MERGED, split,
               class_map=PUBLIC_CLASS_MAP['ramp'], prefix='ramp')

print("\n=== pothole (pothole→8) ===")
for split in ['train', 'valid', 'test']:
    copy_split('/content/datasets/pothole', MERGED, split,
               class_map=PUBLIC_CLASS_MAP['pothole'], prefix='pothole')

print("\n=== 병합 결과 ===")
for split in ['train', 'valid', 'test']:
    imgs = list(Path(f"{MERGED}/{split}/images").glob('*'))
    lbls = list(Path(f"{MERGED}/{split}/labels").glob('*.txt'))
    print(f"{split}: 이미지 {len(imgs)}장 / 라벨 {len(lbls)}개")

=== NoonGil ===
  [train] 780장 완료
  [valid] 74장 완료
  [test] 38장 완료

=== ramp (ramp→9, stairs→10) ===
  [train] 74장 완료
  [valid] 27장 완료
  [test] 11장 완료

=== pothole (pothole→8) ===
  [train] 288장 완료
  [valid] 84장 완료
  [test] 40장 완료

=== 병합 결과 ===
train: 이미지 1142장 / 라벨 1043개
valid: 이미지 185장 / 라벨 156개
test: 이미지 89장 / 라벨 74개


In [6]:
import yaml

data_yaml = {
    'path': MERGED,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 14,
    'names': [
        'bench', 'bicycle', 'bollard', 'clothing_bin', 'cone',
        'electric_scooter', 'fire_hydrant', 'motorcycle',
        'pavement_damage', 'ramp', 'step', 'street_light', 'trash', 'tree'
    ]
}

yaml_path = '/content/datasets/merged/data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, allow_unicode=True, default_flow_style=False)

shutil.copy(yaml_path, f"{DRIVE_BASE}/v8_no_curb_data.yaml")
print("data.yaml 생성 완료")
!cat {yaml_path}

data.yaml 생성 완료
names:
- bench
- bicycle
- bollard
- clothing_bin
- cone
- electric_scooter
- fire_hydrant
- motorcycle
- pavement_damage
- ramp
- step
- street_light
- trash
- tree
nc: 14
path: /content/datasets/merged
test: test/images
train: train/images
val: valid/images


In [7]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')  # s: small, 균형잡힌 크기

results = model.train(
    data='/content/datasets/merged/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    workers=2,
    project=f"{DRIVE_BASE}/runs",
    name='v8_no_curb',
    exist_ok=True,
    patience=20,   # early stopping
)

print("학습 완료")

Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/merged/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=v8_no_curb, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

In [8]:
import pandas as pd

results_dir = f"{DRIVE_BASE}/runs/v8_no_curb"
best_pt = f"{results_dir}/weights/best.pt"

if os.path.exists(best_pt):
    shutil.copy(best_pt, f"{DRIVE_BASE}/v8_no_curb_best.pt")
    print("best.pt 드라이브 저장 완료")

csv_path = f"{results_dir}/results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(df[['epoch',
              'metrics/mAP50(B)',
              'metrics/mAP50-95(B)']].tail(10))

best.pt 드라이브 저장 완료
    epoch  metrics/mAP50(B)  metrics/mAP50-95(B)
40     41           0.84152              0.49963
41     42           0.84263              0.48800
42     43           0.83506              0.48425
43     44           0.84532              0.49571
44     45           0.84590              0.48617
45     46           0.85132              0.49325
46     47           0.85065              0.49225
47     48           0.85033              0.48996
48     49           0.84804              0.49326
49     50           0.85202              0.49504


### YOLOv12 epoch 100으로 늘려서 재학습 진행

In [9]:
import os
os.chdir('/content')

!git clone https://github.com/sunsmarterjie/yolov12.git
os.chdir('/content/yolov12')

# flash_attn, onnxruntime-gpu 제외하고 설치
!grep -v "flash_attn\|onnxruntime-gpu" requirements.txt | pip install -r /dev/stdin -q
!pip install -e . -q

print("YOLOv12 설치 완료")

Cloning into 'yolov12'...
remote: Enumerating objects: 1173, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1173 (delta 0), reused 0 (delta 0), pack-reused 1172 (from 2)
Receiving objects: 100% (1173/1173), 1.95 MiB | 19.22 MiB/s, done.
Resolving deltas: 100% (531/531), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 89.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement onnxruntime==1.15.1 (from versions: 1.17.0, 1.17.1, 1.17.3, 1.18.0, 1.18.1, 1.19.0, 1.19.2, 1.20.0, 1.20.1, 1.21.0, 1.21.1, 1.22.0, 1.22.1, 1.23.0, 1.23.1, 1.23.2, 1.24.1, 1.24.2, 1.24.3, 1.24.4, 1.25.0, 1.25.1, 1.26.0)
ERRO

In [10]:
import os
os.chdir('/content/yolov12')

!yolo detect train \
  data="/content/datasets/merged/data.yaml" \
  model=yolov12s.pt \
  epochs=100 \
  imgsz=640 \
  batch=16 \
  workers=2 \
  patience=30 \
  project="/content/drive/MyDrive/NoonGil/runs" \
  name=v12_no_curb_ep100 \
  exist_ok=True

print("학습 완료")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/yolov12/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
100% 17.8M/17.8M [00:00<00:00, 52.8MB/s]
New https://pypi.org/project/ultralytics/8.4.56 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.63 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov12s.pt, data=/content/datasets/merged/data.yaml, epochs=100, time=None, patience=30, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=2, project=/content/drive/MyDrive/NoonGil/runs, name=v12_no_curb_ep100, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=T

In [11]:
import pandas as pd

results_dir = f"{DRIVE_BASE}/runs/v12_no_curb_ep100"
best_pt = f"{results_dir}/weights/best.pt"

if os.path.exists(best_pt):
    shutil.copy(best_pt, f"{DRIVE_BASE}/v12_no_curb_ep100_best.pt")
    print("best.pt 드라이브 저장 완료")

csv_path = f"{results_dir}/results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(df[['epoch',
              'metrics/mAP50(B)',
              'metrics/mAP50-95(B)']].tail(10))

best.pt 드라이브 저장 완료
    epoch  metrics/mAP50(B)  metrics/mAP50-95(B)
90     91           0.83384              0.50384
91     92           0.83569              0.50203
92     93           0.84181              0.50775
93     94           0.85535              0.52531
94     95           0.84152              0.50645
95     96           0.84400              0.49772
96     97           0.84526              0.51099
97     98           0.83946              0.50628
98     99           0.84591              0.51060
99    100           0.85036              0.50987
